In [ ]:
import kagglehub
import pandas as pd
import numpy as np
import re

import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Descargar última versión
# path = kagglehub.dataset_download("zynicide/wine-reviews")
# print("Path to dataset files:", path)

In [ ]:
# Leer información
df_wine = pd.read_csv('data/winemag-data-130k-v2.csv')
df_wine.head()

In [ ]:
# Extraer con regex el año del título
target = 'title'
year = r'(19\d{2}|20\d{2})'
df_wine['year'] = df_wine[target].str.extract(year, expand=False)

df_wine['year'] = pd.to_numeric(
    df_wine['year'],
    errors='coerce'
)

print(df_wine[['title', 'year']].head(10))

In [ ]:
# Eliminar datos por debajo de 2004
mask_year = (df_wine['year'] >= 2004)
df_wine = df_wine[mask_year].copy()

In [ ]:
# Calcular porcentaje de nulos
print('Datos nulos:')
null_data = (df_wine.isnull().sum() / len(df_wine)) * 100
print(null_data[null_data > 0].sort_values(ascending=False))

In [ ]:
# Seleccionar columnas de interés
cols = [
    'country',
    'description',
    'designation',
    'points',
    'price',
    'province',
    'region_1',
    'title',
    'variety',
    'year'
]

df_wine = df_wine[cols]
df_wine.head()

In [ ]:
# Eliminar datos nulos
cols=[
    'year',
    'variety',
    'price',
    'designation'
]

df_wine = df_wine.dropna(subset=cols).copy()
df_wine = df_wine.reset_index(drop=True)

In [ ]:
# Calcular porcentaje de nulos
print('Datos nulos:')
null_data = (df_wine.isnull().sum() / len(df_wine)) * 100
print(null_data[null_data > 0].sort_values(ascending=False))

In [ ]:
# Se comprueba cuantos vinos existen por país
country_counts = df_wine['country'].value_counts().head(15).reset_index()
country_counts.columns = ['country', 'count']

plt.figure(figsize=(12, 8))

sns.barplot(
    data=country_counts,
    x='count',
    y='country'
)

In [ ]:
# Se hace muestreo aleatorio de los vinos para acotarse al tamaño máximo del dataset
countries_list = df_wine['country'].value_counts().head(10).index.tolist()

# Para estimar el total de registros por país
max_regs = 10000 // len(countries_list)

sampled_dfs = []

for country in countries_list:
    df_temp = df_wine[df_wine['country'] == country]

    if len(df_temp) > max_regs:
        sampled_dfs.append(df_temp.sample(n=max_regs, random_state=42))
    else:
        sampled_dfs.append(df_temp)

df_wine = pd.concat(sampled_dfs).reset_index(drop=True)

In [ ]:
# Se comprueba cuantos vinos existen por país tras la transformación
country_counts = df_wine['country'].value_counts().head(15).reset_index()
country_counts.columns = ['country', 'count']

plt.figure(figsize=(12, 8))

sns.barplot(
    data=country_counts,
    x='count',
    y='country'
)

In [ ]:
# Traducción al español
df_wine['country'] = df_wine['country'].replace(
    {
        'US':               'Estados Unidos',
        'Italy':            'Italia',
        'France':           'Francia',
        'Spain':            'España',
        'Germany':          'Alemania',
        'South Africa':     'Sudáfrica',
        'New Zealand':      'Nueva Zelanda',
        'Greece':           'Grecia',
    }
)


In [ ]:
def clasificar_es(p):
    if p >= 98: return '6/6 Inmejorable'
    if p >= 94: return '5/6 Superior'
    if p >= 90: return '4/6 Excelente'
    if p >= 87: return '3/6 Muy Bueno'
    if p >= 83: return '2/6 Bueno'
    return '1/6 Aceptable'

df_wine['categoria'] = df_wine['points'].apply(clasificar_es)

# Gráfico 1 (Barras)
df_graph1 = df_wine.groupby(['country', 'categoria']).size().reset_index(name='Cantidad')

df_graph1.columns = ['País', 'Calidad', 'Cantidad']
df_graph1.to_csv('graph_1.csv', index=False, sep=',')

# Gráfico 2 (Precio)
df_graph3 = df_wine.dropna(subset=['price'])

df_graph2 = df_graph3.groupby(['country', 'categoria'])['price'].mean().reset_index()
df_graph2['price'] = df_graph2['price'].round(2)

df_graph2.columns = ['País', 'Calidad', 'Precio promedio ($)']
df_graph2.to_csv('graph_2.csv', index=False, sep=',')

In [ ]:
# Gráfico 4 (tarjetas)
bins = [0, 10, 20, 50, 100, 500, 1500]
labels = ['Hasta 10€', 'Hasta 20€', 'Hasta 50€', 'Hasta 100€', 'Hasta 500€', 'Hasta 1500€']

df_wine['price_range'] = pd.cut(df_wine['price'], bins=bins, labels=labels)

df_top = df_wine.sort_values(by=['points', 'price'], ascending=[False, True])
df_top = df_top.groupby('price_range', observed=True).head(3)

df_graph4 = pd.DataFrame()
df_graph4['Nombre'] = df_top['variety']
df_graph4['Imagen'] = ""
df_graph4['País'] = df_top['country']
df_graph4['Rango de Precio'] = df_top['price_range']
df_graph4['Puntuación'] = df_top['points'].astype(str) + " pts"
df_graph4['Precio'] = df_top['price'].astype(str) + "€"

df_graph4.to_csv('graph_4.csv', index=False)

In [ ]:
# Gráfico 5 (mapa)
df_graph5 = pd.read_csv('data/winemag-data-130k-v2.csv')
df_graph5.head()

# Mapeo de nombres
mapeo_oficial = {
    'US': 'United States of America',
    'England': 'United Kingdom',
    'Macedonia': 'Republic of Macedonia',
    'Germany': 'Germany',
    'Italy': 'Italy',
    'France': 'France',
    'Spain': 'Spain',
    'Portugal': 'Portugal',
    'Australia': 'Australia',
    'Austria': 'Austria',
    'Argentina': 'Argentina',
    'Chile': 'Chile',
    'South Africa': 'South Africa',
    'New Zealand': 'New Zealand',
    'Israel': 'Israel',
    'Hungary': 'Hungary',
    'Greece': 'Greece',
    'Romania': 'Romania',
    'Mexico': 'Mexico',
    'Canada': 'Canada',
    'Turkey': 'Turkey',
    'Czech Republic': 'Czech Republic',
    'Slovenia': 'Slovenia',
    'Luxembourg': 'Luxembourg',
    'Croatia': 'Croatia',
    'Georgia': 'Georgia',
    'Uruguay': 'Uruguay',
    'Lebanon': 'Lebanon',
    'Serbia': 'Serbia',
    'Brazil': 'Brazil',
    'Moldova': 'Moldova',
    'Morocco': 'Morocco',
    'Peru': 'Peru',
    'India': 'India',
    'Bulgaria': 'Bulgaria',
    'Cyprus': 'Cyprus',
    'Armenia': 'Armenia',
    'Switzerland': 'Switzerland',
    'Bosnia and Herzegovina': 'Bosnia and Herzegovina',
    'Ukraine': 'Ukraine',
    'Slovakia': 'Slovakia',
    'China': 'China',
    'Egypt': 'Egypt'
}

df_map = df_graph5.dropna(subset=['country']).copy()
df_map['Country name'] = df_map['country'].map(mapeo_oficial).fillna(df_map['country'])

df_export = df_map.groupby('Country name')['variety'].nunique().reset_index()
df_export.columns = ['Country name', 'Variety count']

df_export.to_csv('graph_5.csv', index=False)